In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/samsumdata.zip"  # 👈 change if your uploaded file name is different
extract_dir = "/content/samsum_dataset"

# Create directory and extract
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"✅ Extracted SAMSum dataset to: {extract_dir}")

# Show a few files
print("\n📂 Sample contents:")
for root, dirs, files in os.walk(extract_dir):
    for f in files[:10]:
        print("   └──", os.path.join(root, f))
    break  # only show top-level folder


✅ Extracted SAMSum dataset to: /content/samsum_dataset

📂 Sample contents:
   └── /content/samsum_dataset/data.zip
   └── /content/samsum_dataset/samsum-train.csv
   └── /content/samsum_dataset/samsum-test.csv
   └── /content/samsum_dataset/samsum-validation.csv


In [ ]:
import shutil

# Source (local) and destination (Drive) paths
src_path = "/content/samsum_dataset"   # 👈 replace with your actual extracted folder name
dst_path = "/content/drive/MyDrive/samsum_dataset"

# Copy folder to Google Drive
shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
print(f"✅ SAMSum dataset saved to Drive at: {dst_path}")


✅ SAMSum dataset saved to Drive at: /content/drive/MyDrive/samsum_dataset


In [ ]:
from datasets import load_dataset

# Load directly from your Drive path
dataset = load_dataset(
    "csv",
    data_files={
        "train": "/content/drive/MyDrive/samsum_dataset/samsum-train.csv",
        "validation": "/content/drive/MyDrive/samsum_dataset/samsum-validation.csv",
        "test": "/content/drive/MyDrive/samsum_dataset/samsum-test.csv",
    }
)

# Quick check
print(dataset)
print("\n📄 Example from training set:")
print(dataset["train"][0])


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

📄 Example from training set:
{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}


In [ ]:
from transformers import pipeline

# Create summarization pipeline using t5-small
summarizer = pipeline("summarization", model="t5-small")

# Pick one sample dialogue
sample_text = dataset["test"][0]["dialogue"]

print("🗨️ Original Dialogue:\n", sample_text)
print("\n---\n")

# Summarize
summary = summarizer(
    "summarize: " + sample_text,
    max_length=60,
    min_length=10,
    do_sample=False
)[0]["summary_text"]

print("✨ Generated Summary:\n", summary)


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Device set to use cuda:0


🗨️ Original Dialogue:
 Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

---



Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✨ Generated Summary:
 Amanda: Lemme check Hannah: file_gif> Amanda: Sorry, can't find it . he called her last time we were at the park together .


In [ ]:
from transformers import pipeline

# Use a larger summarizer
summarizer_bart = pipeline("summarization", model="facebook/bart-large-cnn")

# Test on the same example
sample_text = dataset["test"][0]["dialogue"]

print("🗨️ Original Dialogue:\n", sample_text)
print("\n---\n")

summary_bart = summarizer_bart(
    sample_text,
    max_length=80,
    min_length=10,
    do_sample=False
)[0]["summary_text"]

print("✨ BART Summary:\n", summary_bart)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


🗨️ Original Dialogue:
 Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

---

✨ BART Summary:
 Hannah asks Amanda for Betty's number. Amanda can't find it. Hannah asks Larry. Amanda asks Larry to call Betty.


In [ ]:
# ------------------------------------------------------------
# 1️⃣ Imports
# ------------------------------------------------------------
from datasets import load_from_disk, load_dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
import torch

# ------------------------------------------------------------
# 2️⃣ Load SAMSum dataset (from Drive or local)
# ------------------------------------------------------------
from datasets import load_dataset

# ------------------------------------------------------------
# 2️⃣ Load SAMSum dataset from your CSV files in Drive
# ------------------------------------------------------------
dataset = load_dataset(
    "csv",
    data_files={
        "train": "/content/drive/MyDrive/samsum_dataset/samsum-train.csv",
        "validation": "/content/drive/MyDrive/samsum_dataset/samsum-validation.csv",
        "test": "/content/drive/MyDrive/samsum_dataset/samsum-test.csv",
    },
)

print(dataset)


# ------------------------------------------------------------
# 3️⃣ Load model & tokenizer
# ------------------------------------------------------------
model_name = "facebook/bart-large-cnn"  # You can swap with "facebook/bart-base" for faster training
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# ------------------------------------------------------------
# 4️⃣ Preprocessing function
# ------------------------------------------------------------
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = [str(d) if d else "" for d in examples["dialogue"]]
    targets = [str(s) if s else "" for s in examples["summary"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length",
    )["input_ids"]

    model_inputs["labels"] = labels
    return model_inputs

# ------------------------------------------------------------
# 5️⃣ Tokenize dataset
# ------------------------------------------------------------
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["dialogue", "summary", "id"],
)
print("✅ Tokenization successful!")

# ------------------------------------------------------------
# 6️⃣ Data collator
# ------------------------------------------------------------
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# ------------------------------------------------------------
# 7️⃣ Training arguments
# ------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/bart_samsum_checkpoints",  # 📂 save in Drive
    eval_strategy="epoch",          # evaluate every epoch
    save_strategy="epoch",                # save every epoch
    learning_rate=5e-5,
    per_device_train_batch_size=4,        # adjust if out of memory
    per_device_eval_batch_size=4,
    num_train_epochs=2,                   # train for 2 epochs
    weight_decay=0.01,
    save_total_limit=2,                   # keep last 2 checkpoints
    fp16=torch.cuda.is_available(),       # mixed precision for faster training
    logging_dir="/content/drive/MyDrive/bart_logs",
    logging_steps=50,
    push_to_hub=False,
    report_to="none",                     # disable WANDB
)

# ------------------------------------------------------------
# 8️⃣ Trainer setup
# ------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# ------------------------------------------------------------
# 9️⃣ Train!
# ------------------------------------------------------------
trainer.train()

# ------------------------------------------------------------
# 🔟 Save final model
# ------------------------------------------------------------
trainer.save_model("/content/drive/MyDrive/bart_samsum_final")
tokenizer.save_pretrained("/content/drive/MyDrive/bart_samsum_final")

print("✅ Training complete and model saved!")

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

✅ Tokenization successful!


/tmp/ipython-input-2013684167.py:105: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.288300,0.316680
2,0.191900,0.317297


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


✅ Training complete and model saved!


In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

model_path = "/content/drive/MyDrive/bart_samsum_final"
tokenizer = BartTokenizer.from_pretrained(model_path)
model = BartForConditionalGeneration.from_pretrained(model_path).to("cuda")

dialogue = """Hannah: Hey, do you have Betty's number?
Amanda: Lemme check.
Amanda: Sorry, can't find it.
Amanda: Ask Larry.
Hannah: I don't know him well.
Amanda: Don't be shy, he's nice.
Hannah: Okay, I’ll try."""

# Prepare inputs
inputs = tokenizer(
    dialogue,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
).to("cuda")

# Generate summary
summary_ids = model.generate(
    **inputs,
    max_length=60,
    min_length=10,
    num_beams=4,
    length_penalty=2.0,
    no_repeat_ngram_size=3,
    early_stopping=True,
)

# Decode
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("✨ Summary:", summary)


✨ Summary: Amanda doesn't have Betty's number. Hannah will ask Larry for it.


In [ ]:
from evaluate import load
import numpy as np

metric = load("rouge")

def evaluate_model(model, tokenizer, dataset, num_samples=100):
    model.eval()
    predictions, references = [], []
    for example in dataset.select(range(num_samples)):
        inputs = tokenizer(example["dialogue"], return_tensors="pt", truncation=True, padding=True, max_length=512).to("cuda")
        summary_ids = model.generate(
            **inputs,
            max_length=60,
            min_length=10,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
        pred = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        predictions.append(pred)
        references.append(example["summary"])
    results = metric.compute(predictions=predictions, references=references)
    return {key: value for key, value in results.items()}

# Evaluate on test split
rouge_scores = evaluate_model(model, tokenizer, dataset["test"], num_samples=200)
print("\n📊 ROUGE Evaluation Results:")
for key, value in rouge_scores.items():
    print(f"{key}: {value:.4f}")


📊 ROUGE Evaluation Results:
rouge1: 0.4894
rouge2: 0.2491
rougeL: 0.4001
rougeLsum: 0.4016


In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration
import torch

summary_model_path = "/content/drive/MyDrive/bart_samsum_final"  # your saved path
tokenizer = BartTokenizer.from_pretrained(summary_model_path)
model = BartForConditionalGeneration.from_pretrained(summary_model_path)


/usr/local/lib/python3.12/dist-packages/transformers/models/bart/configuration_bart.py:177: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


In [ ]:
from transformers import pipeline
story_generator = pipeline("text2text-generation", model="google/flan-t5-large")


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [ ]:
def chat_to_story(chat_text, mood="drama"):
    # Step 1: Summarize
    inputs = tokenizer(chat_text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    summary_ids = model.generate(**inputs, max_length=128)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # Step 2: Turn into story prompt
    prompt = f"Write a {mood} short story based on this summary: {summary}"

    # Step 3: Generate story
    story = story_generator(prompt, max_length=300, num_return_sequences=1)[0]["generated_text"]

    return summary, story


In [ ]:
!ls /content/drive/MyDrive/bart_samsum_checkpoints


checkpoint-3683  checkpoint-7366


In [ ]:
!ls /content/drive/MyDrive/bart_samsum_checkpoints/checkpoint-7366


config.json		rng_state.pth		 tokenizer_config.json
generation_config.json	scaler.pt		 trainer_state.json
merges.txt		scheduler.pt		 training_args.bin
model.safetensors	special_tokens_map.json  vocab.json
